In [13]:
import sempy.fabric as fabric
import pandas as pd
from pyspark.sql.functions import lit
from datetime import datetime

StatementMeta(, 1c396d74-f15a-4fdb-9839-e373d15a4758, 17, Finished, Available, Finished)

In [14]:
%run 2. Parameters


StatementMeta(, 1c396d74-f15a-4fdb-9839-e373d15a4758, 18, Finished, Available, Finished)

In [15]:
# Get all tables
tables_info = fabric.evaluate_dax(datasetId, dax_query_tables, workspace=workspaceId)

# Get all partitions
partitions_info = fabric.evaluate_dax(datasetId, dax_query_partitions, workspace=workspaceId)

# Remove brackets and add prefixes to column names
tables_info.columns = ['t_' + col.replace('[', '').replace(']', '') for col in tables_info.columns]
partitions_info.columns = ['p_' + col.replace('[', '').replace(']', '') for col in partitions_info.columns]

# First, let's see the column names
print("Tables columns:", tables_info.columns.tolist())
print("Partitions columns:", partitions_info.columns.tolist())

# Perform left join
combined_df = partitions_info.merge(
    tables_info, 
    left_on='p_TableID',
    right_on='t_ID',
    how='left'
)

display(combined_df)

StatementMeta(, 1c396d74-f15a-4fdb-9839-e373d15a4758, 19, Finished, Available, Finished)

Tables columns: ['t_ID', 't_ModelID', 't_Name', 't_DataCategory', 't_Description', 't_IsHidden', 't_TableStorageID', 't_ModifiedTime', 't_StructureModifiedTime', 't_SystemFlags', 't_ShowAsVariationsOnly', 't_IsPrivate', 't_DefaultDetailRowsDefinitionID', 't_AlternateSourcePrecedence', 't_RefreshPolicyID', 't_CalculationGroupID', 't_ExcludeFromModelRefresh', 't_LineageTag', 't_SourceLineageTag', 't_SystemManaged']
Partitions columns: ['p_ID', 'p_TableID', 'p_Name', 'p_Description', 'p_DataSourceID', 'p_QueryDefinition', 'p_State', 'p_Type', 'p_PartitionStorageID', 'p_Mode', 'p_DataView', 'p_ModifiedTime', 'p_RefreshedTime', 'p_SystemFlags', 'p_ErrorMessage', 'p_RetainDataTillForceCalculate', 'p_RangeStart', 'p_RangeEnd', 'p_RangeGranularity', 'p_RefreshBookmark', 'p_QueryGroupID', 'p_ExpressionSourceID', 'p_MAttributes']


SynapseWidget(Synapse.DataFrame, 6c73304b-9d76-4697-b847-e9951f77d09b)

In [16]:
from sempy import fabric

# Use the combined_df created in the previous code block
r_partitions_df = combined_df.copy()  # Make a copy to avoid modifying the original

# Add columns to your pandas DataFrame
r_partitions_df['workspace_id'] = workspaceId
r_partitions_df['dataset_id'] = datasetId
r_partitions_df['database_name'] = semantic_model_name

# Automatically replace spaces with underscores in all column names
r_partitions_df.columns = r_partitions_df.columns.str.replace(' ', '_').str.upper()

# Drop columns with void/null datatypes
r_partitions_df = r_partitions_df.dropna(axis=1, how='all')

# Alternatively, explicitly drop the problematic column
# r_partitions_df = r_partitions_df.drop(columns=['SORT_BY_COLUMN'], errors='ignore')

# Define table name
table_name = "r_partitions"

# Drop the table if it exists to clear any metadata issues
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Convert to Spark DataFrame and write
spark_r_partitions_df = spark.createDataFrame(r_partitions_df)
spark_r_partitions_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(table_name)

print(f"Successfully wrote {spark_r_partitions_df.count()} rows to table '{table_name}'")

# Now query and display
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1000")
display(df)

StatementMeta(, 1c396d74-f15a-4fdb-9839-e373d15a4758, 20, Finished, Available, Finished)

Successfully wrote 8 rows to table 'r_partitions'


SynapseWidget(Synapse.DataFrame, e7076ad9-176a-439e-9be8-a6d1ddb3e102)

In [17]:
df = spark.sql("SELECT * FROM SMD_LH.dbo.r_partitions LIMIT 1000")
display(df)

StatementMeta(, 1c396d74-f15a-4fdb-9839-e373d15a4758, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c4300e32-e64b-4b4c-8e1c-6161ae75679f)

In [19]:
# Create Spark DataFrame from the filtered Pandas DataFrame
spark_df = spark.createDataFrame(combined_df)

# Add columns to your pandas DataFrame
r_partitions_df['workspace_id'] = workspaceId
r_partitions_df['dataset_id'] = datasetId
r_partitions_df['database_name'] = semantic_model_name

# Convert all column names to uppercase
for col_name in spark_df.columns:
    spark_df = spark_df.withColumnRenamed(col_name, col_name.upper())

# Add database name and timestamp column using Spark syntax
spark_df = spark_df.withColumn("DATABASE_TIMESTAMP", 
                                lit(f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"))

# Define table name
table_name = "table_partitions"

# Determine version number based on existing data
if spark.catalog.tableExists(table_name):
    # Read existing table to find the max version for this semantic model
    existing_df = spark.read.table(table_name)
    
    # Filter for current semantic model and get max version
    max_version = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()[0]['max_ver']
    
    # Increment version, or start at 1 if this model hasn't been added yet
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
spark_df = spark_df.withColumn('DATABASE_VERSION', lit(f"{semantic_model_name} | V{next_version}"))

# Write using the determined mode
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

display(spark_df)

StatementMeta(, 1c396d74-f15a-4fdb-9839-e373d15a4758, 23, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 64776744-a817-4176-9ca4-1ec9c35a8b7d)